# Compliance Q&A: a Retrieval-Augmented Generation project

Welcome! In this project you will build a **RAG application** that lets employees ask questions
about company policy, such as *"How many days of annual leave do I get?"* or *"Can I accept a CHF
120 gift from a vendor?"*, and get answers grounded in a real document set.

You will implement the **basics of a RAG system** yourself: chunking, embeddings, keyword search,
fusion, and the final answer step. The surrounding scaffolding (data models, the vector store, the
AI clients, a web app) is provided, so you can focus on the ideas that make RAG work.

After each function you implement, you will run a small **test cell** that checks it against a set
of expected behaviours. Once every piece is in place, you will run the **whole pipeline start to
finish** and watch it answer a real question.

## How this notebook works

The project ships as a normal Python repo. Rather than editing those files directly, you will write
each function **here, in the notebook**, and one short line wires it into the real project code so
the rest of the pipeline (and the tests) immediately use your version. At the end of each part an
**export cell** writes your functions to a file you can drop into the repo to run the full app.

## Roadmap

- **Part 1, Ingestion:** turn documents into searchable, embedded chunks (chunking, embeddings,
  metadata).
- **Part 2, RAG core:** answer a question (keyword search, embedding search, fusion, reranking, and
  finally calling the LLM with the retrieved context).

Each step is: a short explanation, then a function for **you** to implement, then a **test cell**
that tells you whether it works. Look for the 🎯 marker, it always points at something you need to
do.

### The whole system at a glance

In [ ]:
#@title 📊 Diagram: the whole system at a glance (run me to view) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div style="text-align:center;margin:14px 0;font-family:ui-sans-serif,system-ui,sans-serif">
<svg viewBox="0 0 860 320" width="100%" style="max-width:860px">
<defs>
  <marker id="ah1" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
    <path d="M0,0 L8,3 L0,6 Z" fill="#647084"/>
  </marker>
  <linearGradient id="g1" x1="0" y1="0" x2="0" y2="1">
    <stop offset="0%" stop-color="#0f766e" stop-opacity="0.12"/>
    <stop offset="100%" stop-color="#0f766e" stop-opacity="0.02"/>
  </linearGradient>
  <linearGradient id="g2" x1="0" y1="0" x2="0" y2="1">
    <stop offset="0%" stop-color="#b45309" stop-opacity="0.12"/>
    <stop offset="100%" stop-color="#b45309" stop-opacity="0.02"/>
  </linearGradient>
</defs>

<text x="20" y="26" font-size="13" font-weight="700" fill="#334155">1. INGEST (build the searchable store)</text>
<g font-family="ui-sans-serif,system-ui,sans-serif">
  <rect x="20" y="42" width="180" height="70" rx="14" fill="url(#g1)" stroke="#0f766e" stroke-width="1.6"/>
  <text x="110" y="72" text-anchor="middle" font-size="22">📄</text>
  <text x="110" y="98" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Load documents</text>

  <rect x="230" y="42" width="180" height="70" rx="14" fill="url(#g1)" stroke="#0f766e" stroke-width="1.6"/>
  <text x="320" y="72" text-anchor="middle" font-size="22">✂️</text>
  <text x="320" y="98" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Chunk</text>

  <rect x="440" y="42" width="180" height="70" rx="14" fill="url(#g1)" stroke="#0f766e" stroke-width="1.6"/>
  <text x="530" y="72" text-anchor="middle" font-size="22">🔢</text>
  <text x="530" y="98" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Embed</text>

  <rect x="650" y="42" width="180" height="70" rx="14" fill="url(#g2)" stroke="#b45309" stroke-width="1.6"/>
  <text x="740" y="72" text-anchor="middle" font-size="22">🗄️</text>
  <text x="740" y="98" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Store in Qdrant</text>

  <path d="M200,77 L230,77" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah1)"/>
  <path d="M410,77 L440,77" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah1)"/>
  <path d="M620,77 L650,77" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah1)"/>
</g>

<text x="20" y="168" font-size="13" font-weight="700" fill="#334155">2. ASK (answer a question)</text>
<g font-family="ui-sans-serif,system-ui,sans-serif">
  <rect x="20" y="184" width="150" height="70" rx="14" fill="#ffffff" stroke="#334155" stroke-width="1.6"/>
  <text x="95" y="214" text-anchor="middle" font-size="22">❓</text>
  <text x="95" y="240" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Question</text>

  <rect x="205" y="184" width="200" height="70" rx="14" fill="url(#g1)" stroke="#0f766e" stroke-width="1.6"/>
  <text x="305" y="210" text-anchor="middle" font-size="20">🔎</text>
  <text x="305" y="232" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Hybrid search</text>
  <text x="305" y="248" text-anchor="middle" font-size="10.5" fill="#647084">keyword + embedding, fused</text>

  <rect x="440" y="184" width="150" height="70" rx="14" fill="url(#g1)" stroke="#0f766e" stroke-width="1.6"/>
  <text x="515" y="214" text-anchor="middle" font-size="22">🧠</text>
  <text x="515" y="240" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">LLM</text>

  <rect x="625" y="184" width="205" height="70" rx="14" fill="url(#g2)" stroke="#b45309" stroke-width="1.6"/>
  <text x="695" y="214" text-anchor="middle" font-size="22">✅</text>
  <text x="727" y="240" text-anchor="middle" font-size="12.5" font-weight="700" fill="#1f2933">Answer</text>
  <text x="727" y="253" text-anchor="middle" font-size="10.5" fill="#647084">+ citations</text>

  <path d="M170,219 L205,219" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah1)"/>
  <path d="M405,219 L440,219" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah1)"/>
  <path d="M590,219 L625,219" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah1)"/>
</g>

<path d="M740,112 L740,140 L305,140 L305,184" stroke="#647084" stroke-width="1.6" fill="none"
      marker-end="url(#ah1)" stroke-dasharray="5 4"/>
<text x="522" y="156" text-anchor="middle" font-size="10.5" font-weight="600" fill="#647084">
  the stored chunks are what search looks through
</text>

<text x="20" y="298" font-size="11" fill="#647084">
  Part 1 builds the top row. Part 2 builds the bottom row. They meet at the vector store.
</text>
</svg>
</div>
'''))


## 0.1 - Setup

This notebook reads its credentials from **Colab Secrets**, never paste keys into a cell. Open the
**🔑 key icon** in the left sidebar ("Secrets") and add the following, toggling **"Notebook access"
ON** for each:

| Secret name | What it's for | Required? |
|---|---|---|
| `OPENROUTER_API_KEY` | the *optional* live demo at the end | optional |

Now run the cell below. In **Colab** it clones the project (the `REPO_BRANCH` branch), installs
dependencies, and puts the code on the import path. Running **locally inside the repo**, it just
makes the repo importable.

In [ ]:
#@title ⚙️ Setup: clone the repo and install packages (run me, no need to read) { display-mode: "form" }
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

# If the course gives you a different template URL, change this.
REPO_URL = "https://github.com/eth-fdd-fs26/FDD-WE5-public.git"
# The branch that holds the project files (this notebook + the solutions/ loader).
# Change to "main" once the project has been merged there.
REPO_BRANCH = "project"
REPO_DIR = "FDD-WE5-public"


def colab_secret(name):
    """Read a Colab secret by name; return None if missing or access not granted."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        # --branch pins the correct branch; --single-branch keeps the clone small.
        proc = subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR],
            capture_output=True, text=True,
        )
        if proc.returncode != 0:
            detail = (proc.stderr or proc.stdout or "").strip()
            raise RuntimeError(
                "git clone failed (exit %d):\n%s\n\nCommon causes:\n"
                "  - REPO_BRANCH (%r) doesn't exist on the remote." % (proc.returncode, detail, REPO_BRANCH)
            )
    %pip install -q qdrant-client rank-bm25 numpy openai pypdf rich
    sys.path.insert(0, os.path.abspath(REPO_DIR))
    os.chdir(REPO_DIR)
else:
    # Local: find the repo root (the folder that contains chunking.py), make it
    # importable, and work from there, so it doesn't matter whether you launched
    # the notebook from the repo root or from the notebook/ subfolder.
    root = os.path.abspath(".")
    while root != os.path.dirname(root) and not os.path.exists(os.path.join(root, "chunking.py")):
        root = os.path.dirname(root)
    sys.path.insert(0, root)
    os.chdir(root)

print("Setup complete. Running in Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

## 0.2 - An optional OpenRouter key

Everything in this notebook runs **fully offline** using small mock AI clients, so you do not need
an API key to do the project or pass the tests.

If you *do* want to see the real system answer a question at the end, add your
[OpenRouter](https://openrouter.ai) key as the Colab secret **`OPENROUTER_API_KEY`** (🔑 Secrets
panel, "Notebook access" ON). The cell below loads it from your secrets. Locally, you can instead
run `export OPENROUTER_API_KEY=...` before launching Jupyter. The key is read straight into the
environment and never written into the notebook.

In [ ]:
#@title 🔑 Load your optional OpenRouter key (run me) { display-mode: "form" }
# Pull the OpenRouter key from Colab Secrets (no-op locally / if the secret is absent).
_k = colab_secret("OPENROUTER_API_KEY") if IN_COLAB else None
if _k:
    os.environ["OPENROUTER_API_KEY"] = _k

HAS_KEY = bool(os.environ.get("OPENROUTER_API_KEY"))
print("Live API key available:", HAS_KEY)
print("(Everything below works offline with mocks regardless.)")

## 0.3 - Imports

We import the provided scaffolding once. Notice we reuse the project's own test harness
(`TestSuite`) and offline doubles (`MockEmbedder`, `MockLLM`), the exact same tools the course
maintainers use.

In [ ]:
#@title 📦 Imports (run me, no need to read) { display-mode: "form" }
import inspect
import pathlib

import numpy as np
from rank_bm25 import BM25Okapi

# Provided scaffolding from the repo:
import chunking
import loaders
from ingestion_core import IngestionCore
from db_manager import DBManager
from rag_core import RAGCore
from retrieval_core import RetrievalCore, SearchType, _tokenize
from models import Chunk, Document

# Offline test doubles + the friendly test runner (also from the repo):
from tests.mocks import MockEmbedder, MockLLM
from tests.harness import TestSuite

print("Imports OK. Ready to build the pipeline.")

## 0.4 - Pretty output helpers

To make results easy to read, we render them as small HTML cards instead of plain text. Just run
this cell, you'll call these helpers (`show_documents`, `show_chunks`, `show_results`,
`show_answer`) throughout the notebook.

In [ ]:
#@title 🎨 Pretty output helpers (run me, no need to read) { display-mode: "form" }
from IPython.display import HTML, display

# A small, consistent visual language for outputs in this notebook.
_INK, _MUTE, _TEAL, _AMBER, _LINE, _BG = "#1f2933", "#647084", "#0f766e", "#b45309", "#e3e8ef", "#f8fafc"
_FONT = "ui-sans-serif,system-ui,sans-serif"
_MONO = "ui-monospace,SFMono-Regular,Menlo,monospace"

def _card(inner, accent=_TEAL):
    return (f'<div style="border:1px solid {_LINE};border-left:4px solid {accent};border-radius:10px;'
            f'padding:14px 16px;margin:8px 0;background:#fff;font-family:{_FONT}">{inner}</div>')

def _clip(text, n):
    return (text[:n] + "…") if len(text) > n else text

def show_documents(docs):
    head = (f'<tr style="color:{_MUTE};font-size:11px;text-transform:uppercase;letter-spacing:.05em">'
            f'<th style="text-align:left;padding:6px 10px">Document</th>'
            f'<th style="text-align:left;padding:6px 10px">Source</th>'
            f'<th style="text-align:right;padding:6px 10px">Words</th></tr>')
    rows = "".join(
        f'<tr><td style="padding:6px 10px;border-top:1px solid {_LINE};font-weight:600;color:{_INK}">{d.metadata["document_title"]}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid {_LINE};color:{_MUTE};font-family:{_MONO};font-size:12px">{d.metadata["source"]}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid {_LINE};text-align:right;color:{_INK}">{len(d.text.split())}</td></tr>'
        for d in docs)
    title = f'<div style="font-weight:700;color:{_INK};margin-bottom:6px">📚 {len(docs)} policy documents</div>'
    display(HTML(_card(title + f'<table style="border-collapse:collapse;width:100%">{head}{rows}</table>')))

def show_chunks(chunks, title="Chunks", limit=4):
    items = "".join(
        f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:8px 10px;margin:6px 0">'
        f'<span style="font-family:{_MONO};font-size:11px;color:{_TEAL};font-weight:700">#{c.index}</span> '
        f'<span style="color:{_INK};font-size:13px">{_clip(c.text, 150)}</span></div>'
        for c in chunks[:limit])
    more = f'<div style="color:{_MUTE};font-size:12px">… and {len(chunks)-limit} more</div>' if len(chunks) > limit else ""
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:4px">🧩 {title} · {len(chunks)} total</div>{items}{more}')))

def show_chunking(whole, windows):
    def col(name, chunks, accent):
        items = "".join(
            f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:8px;margin:6px 0;font-size:12px;color:{_INK}">{_clip(ch, 120)}</div>'
            for ch in chunks[:4])
        more = f'<div style="color:{_MUTE};font-size:12px">… {len(chunks)-4} more</div>' if len(chunks) > 4 else ""
        return f'<div style="flex:1"><div style="font-weight:700;color:{accent};margin-bottom:4px">{name} · {len(chunks)} chunk(s)</div>{items}{more}</div>'
    display(HTML(_card(f'<div style="display:flex;gap:16px">{col("whole_document", whole, _AMBER)}{col("sliding_window", windows, _TEAL)}</div>')))

def show_results(results, title="Results"):
    if not results:
        display(HTML(_card("<em>No results.</em>", _AMBER)))
        return
    mx = max(s for _, s in results) or 1.0
    rows = []
    for rank, (c, s) in enumerate(results, 1):
        pct = max(4.0, 100 * s / mx)
        rows.append(
            f'<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 0;border-top:1px solid {_LINE}">'
            f'<div style="color:{_MUTE};font-weight:700;width:20px">{rank}</div><div style="flex:1">'
            f'<div style="font-size:11px;color:{_MUTE};font-family:{_MONO}">{c.metadata.get("source", "?")}</div>'
            f'<div style="color:{_INK};font-size:13px;margin:2px 0 4px">{_clip(c.text, 150)}</div>'
            f'<div style="background:{_BG};border-radius:6px;height:6px;overflow:hidden">'
            f'<div style="width:{pct:.0f}%;height:6px;background:{_TEAL}"></div></div></div>'
            f'<div style="font-family:{_MONO};color:{_TEAL};font-weight:700">{s:.3f}</div></div>')
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:2px">🔎 {title}</div>' + "".join(rows))))

def show_answer(answer, sources):
    src = "".join(
        f'<li style="margin:5px 0;color:{_MUTE};font-size:13px"><code style="color:{_TEAL}">{c.metadata.get("source", "?")}</code> '
        f'<span style="color:{_TEAL};font-weight:700">{s:.3f}</span>: {_clip(c.text, 90)}</li>'
        for c, s in sources)
    body = (f'<div style="font-size:11px;color:{_MUTE};text-transform:uppercase;letter-spacing:.05em">Answer</div>'
            f'<div style="color:{_INK};font-size:15px;line-height:1.55;margin:6px 0 10px">{answer}</div>'
            f'<details><summary style="cursor:pointer;color:{_TEAL};font-weight:700">{len(sources)} sources</summary>'
            f'<ul style="margin:8px 0 0;padding-left:18px">{src}</ul></details>')
    display(HTML(_card(body, _AMBER)))


def show_chunking_comparison(sample, whole, windows):
    """Clearer side-by-side: one blurry whole-document vector vs many focused chunk vectors."""
    n_words = len(sample.text.split())
    whole_preview = _clip(whole[0], 220) if whole else ""
    chunk_items = "".join(
        f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:8px 10px;'
        f'margin:5px 0;font-size:12px;color:{_INK}">{_clip(w, 110)}</div>'
        for w in windows[:4]
    )
    more = (f'<div style="color:{_MUTE};font-size:11.5px">... and {len(windows) - 4} more</div>'
            if len(windows) > 4 else "")
    body = (
        f'<div style="display:flex;align-items:center;justify-content:center;gap:18px;margin-bottom:14px">'
        f'<div style="text-align:center"><div style="font-size:34px;font-weight:800;color:{_AMBER}">1</div>'
        f'<div style="font-size:11px;color:{_MUTE};font-weight:700;text-transform:uppercase">whole_document</div></div>'
        f'<div style="font-size:22px;color:{_MUTE}">vs</div>'
        f'<div style="text-align:center"><div style="font-size:34px;font-weight:800;color:{_TEAL}">{len(windows)}</div>'
        f'<div style="font-size:11px;color:{_MUTE};font-weight:700;text-transform:uppercase">sliding_window</div></div>'
        f'</div>'
        f'<div style="font-size:12px;color:{_MUTE};text-align:center;margin-bottom:14px">'
        f'same {n_words}-word document, split two different ways</div>'
        f'<div style="display:flex;gap:16px">'
        f'<div style="flex:1"><div style="font-weight:700;color:{_AMBER};margin-bottom:6px">'
        f'whole_document: 1 blurry vector</div>'
        f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:10px;'
        f'font-size:12px;color:{_MUTE}">{whole_preview}</div></div>'
        f'<div style="flex:1"><div style="font-weight:700;color:{_TEAL};margin-bottom:6px">'
        f'sliding_window: {len(windows)} focused vectors</div>'
        f'{chunk_items}{more}</div>'
        f'</div>'
    )
    display(HTML(_card(body)))

print("Display helpers ready: show_documents, show_chunks, show_chunking, show_chunking_comparison, show_results, show_answer")

---
# Part 1 - Document ingestion pipeline

Ingestion is the **write side** of RAG. It turns raw documents into searchable chunks:

```
load  ->  CHUNK  ->  EMBED  ->  store
```

1. **load**, read a file into a `Document` (provided), attaching metadata.
2. **chunk**, split the text into focused passages (**you implement this**).
3. **embed**, turn each chunk into a vector (**you wire this up**).
4. **store**, persist the embedded chunks into the vector database (provided).

The ✏️ marked steps below are the ones **you** implement. Let's look at the documents we'll work
with.

In [ ]:
#@title 📊 Diagram: Part 1 flow (run me to view) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 770 140" width="100%" style="max-width:770px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ahp1" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><rect x="20" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="100.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">load</text><text x="100.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">file &#8594; Document</text><rect x="210" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="290.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">CHUNK</text><text x="290.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">Document &#8594; texts</text><rect x="400" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="480.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">EMBED</text><text x="480.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">texts &#8594; vectors</text><rect x="590" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="670.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">store</text><text x="670.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">&#8594; vector DB</text><path d="M180,73 L210,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp1)"/><path d="M370,73 L400,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp1)"/><path d="M560,73 L590,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp1)"/><text x="100" y="118" text-anchor="middle" font-size="10" font-weight="600" fill="#647084">loaders.py</text><text x="290" y="118" text-anchor="middle" font-size="10" font-weight="700" fill="#0f766e">chunking.py &#9999;&#65039; you</text><text x="480" y="118" text-anchor="middle" font-size="10" font-weight="700" fill="#0f766e">ingest_document &#9999;&#65039; you</text><text x="670" y="118" text-anchor="middle" font-size="10" font-weight="600" fill="#647084">db_manager.py</text></svg></div>
'''))


In [ ]:
docs = loaders.load_directory("data")
show_documents(docs)

## 1.1 - Chunking

An embedding model turns a piece of text into **one** vector. Embed a whole 800-word policy as a
single vector and it becomes a blurry average of every idea in it, a question about one clause has
to compete with the noise of the entire document.

The fix is **chunking**: split the document into smaller passages so each idea gets its own vector.
We use a **sliding window** over words, with a little **overlap** so a sentence straddling a
boundary isn't lost.

Play with the sliders below to see how `chunk_size` and `overlap` change the windows:

In [ ]:
#@title 🎚️ Interactive: chunk_size vs overlap (run me, then drag the sliders) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div id="ragw-root" style="font-family:ui-sans-serif,system-ui,sans-serif;border:1px solid #e3e8ef;
     border-radius:12px;padding:16px 20px;background:#f8fafc;max-width:720px;margin:10px auto">
  <div style="display:flex;gap:28px;flex-wrap:wrap;margin-bottom:12px">
    <label style="font-size:12.5px;color:#1f2933;font-weight:700">
      chunk_size: <span id="ragw-cs-val" style="color:#0f766e">4</span>
      <input id="ragw-cs" type="range" min="2" max="8" value="4" step="1" style="vertical-align:middle;margin-left:6px">
    </label>
    <label style="font-size:12.5px;color:#1f2933;font-weight:700">
      overlap: <span id="ragw-ov-val" style="color:#b45309">1</span>
      <input id="ragw-ov" type="range" min="0" max="7" value="1" step="1" style="vertical-align:middle;margin-left:6px">
    </label>
  </div>
  <div id="ragw-formula" style="font-size:12px;color:#334155;font-weight:600;margin-bottom:10px">
    step = chunk_size - overlap = 3
  </div>
  <div id="ragw-warning" style="display:none;color:#b91c1c;font-size:12px;font-weight:700;margin-bottom:8px">
    overlap must be smaller than chunk_size
  </div>
  <svg id="ragw-svg" viewBox="0 0 680 220" width="100%" style="max-width:680px;display:block"></svg>
  <div style="font-size:11px;color:#647084;margin-top:6px">
    w0..w11 stand in for words in a real sentence. Each colored band below is one chunk;
    where bands overlap, those words are shared between consecutive chunks.
  </div>
</div>
<script>
(function() {
  var N = 12;                       // number of demo words
  var colors = ["#0f766e", "#b45309", "#334155", "#7c3aed"];
  var csEl = document.getElementById("ragw-cs");
  var ovEl = document.getElementById("ragw-ov");
  var csVal = document.getElementById("ragw-cs-val");
  var ovVal = document.getElementById("ragw-ov-val");
  var formula = document.getElementById("ragw-formula");
  var warning = document.getElementById("ragw-warning");
  var svg = document.getElementById("ragw-svg");

  function windows(chunkSize, overlap) {
    var step = chunkSize - overlap;
    var out = [];
    for (var start = 0; start < N; start += step) {
      var end = Math.min(start + chunkSize, N);
      if (end <= start) break;
      out.push([start, end]);
      if (start + chunkSize >= N) break;
    }
    return out;
  }

  function redraw() {
    var chunkSize = parseInt(csEl.value, 10);
    var overlap = parseInt(ovEl.value, 10);

    // Guard: overlap must stay smaller than chunk_size, exactly like sliding_window() itself.
    if (overlap >= chunkSize) {
      overlap = chunkSize - 1;
      ovEl.value = overlap;
    }
    ovEl.max = chunkSize - 1;

    csVal.textContent = chunkSize;
    ovVal.textContent = overlap;
    var step = chunkSize - overlap;
    formula.textContent = "step = chunk_size - overlap = " + step;
    warning.style.display = "none";

    var boxW = 50, boxGap = 6, x0 = 10, y0 = 14;
    var svgParts = [];
    for (var i = 0; i < N; i++) {
      var x = x0 + i * (boxW + boxGap);
      svgParts.push('<rect x="' + x + '" y="' + y0 + '" width="' + boxW + '" height="30" rx="6" ' +
                     'fill="#ffffff" stroke="#e3e8ef"/>');
      svgParts.push('<text x="' + (x + boxW / 2) + '" y="' + (y0 + 20) + '" text-anchor="middle" ' +
                     'font-size="12" font-weight="600" fill="#1f2933">w' + i + '</text>');
    }

    var ws = windows(chunkSize, overlap);
    var bandY = y0 + 44;
    for (var c = 0; c < ws.length; c++) {
      var start = ws[c][0], end = ws[c][1];
      var bx = x0 + start * (boxW + boxGap);
      var bw = (end - start) * boxW + (end - start - 1) * boxGap;
      var color = colors[c % colors.length];
      var by = bandY + c * 28;
      svgParts.push('<rect x="' + bx + '" y="' + by + '" width="' + bw + '" height="22" rx="6" ' +
                     'fill="' + color + '" fill-opacity="0.16" stroke="' + color + '" stroke-width="1.4"/>');
      svgParts.push('<text x="' + (bx + bw / 2) + '" y="' + (by + 15.5) + '" text-anchor="middle" ' +
                     'font-size="10.5" font-weight="700" fill="' + color + '">' +
                     'chunk ' + c + '  ·  w' + start + '-w' + (end - 1) + '</text>');
    }

    svg.setAttribute("viewBox", "0 0 680 " + (bandY + ws.length * 28 + 10));
    svg.innerHTML = svgParts.join("");
  }

  csEl.addEventListener("input", redraw);
  ovEl.addEventListener("input", redraw);
  redraw();
})();
</script>
'''))


> **Reflection:** what goes wrong if `overlap` is large relative to `chunk_size`? What if it's 0?

### 🎯 Exercise 1 - implement `sliding_window`

In [ ]:
def sliding_window(text, chunk_size=200, overlap=40):
    """Split `text` into overlapping windows of words."""
    if chunk_size <= 0:
        raise ValueError(f"chunk_size must be positive, got {chunk_size}")
    if overlap < 0:
        raise ValueError(f"overlap must be >= 0, got {overlap}")
    if overlap >= chunk_size:
        raise ValueError(f"overlap ({overlap}) must be smaller than chunk_size ({chunk_size})")

    # 🎯 TODO: split the text into a list of individual words
    # (a plain .split() on whitespace is enough)
    words = ...

    # 🎯 TODO: if there are no words at all, there is nothing to chunk,
    # return an empty list right away
    ...

    # 🎯 TODO: work out how many words to move forward between one chunk and
    # the next, so consecutive chunks end up sharing `overlap` words
    step = ...

    chunks = []
    # 🎯 TODO: walk a starting position forward across the words, `step` words
    # at a time, stopping once you reach the end of the text
    for start in range(...):
        # 🎯 TODO: take the next `chunk_size` words starting at this position
        window = ...

        if not window:
            break

        # 🎯 TODO: join this window's words back into one string with single
        # spaces, and add it to the list of chunks
        ...

        # 🎯 TODO: if this window already reached the last word, stop here so
        # you don't emit one final chunk that's fully contained in the last one
        if ...:
            break

    return chunks


Patch it onto the module so the rest of the pipeline picks it up:

In [ ]:
chunking.sliding_window = sliding_window
print("Patched chunking.sliding_window ->", chunking.sliding_window.__name__)

In [ ]:
suite = TestSuite("Part 1.1: sliding_window")

@suite.case("sliding_window", "overlap shares words across consecutive windows")
def _():
    assert sliding_window("a b c d e f", chunk_size=4, overlap=1) == ["a b c d", "d e f"]

@suite.case("sliding_window", "text shorter than the window -> a single chunk")
def _():
    assert sliding_window("a b", chunk_size=4, overlap=1) == ["a b"]

@suite.case("sliding_window", "empty text -> no chunks")
def _():
    assert sliding_window("   ", chunk_size=4, overlap=1) == []

@suite.case("sliding_window", "overlap >= chunk_size is rejected")
def _():
    try:
        sliding_window("a b c", chunk_size=2, overlap=2)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass

suite.run()

**See why chunking matters.** Compare the "no chunking" baseline against your sliding window on a
real policy, notice how many focused passages you get instead of one giant blob.

In [ ]:
sample = next(d for d in docs if "leave" in d.metadata["source"])
whole = chunking.chunk_text(sample.text, strategy="whole_document")
windows = chunking.chunk_text(sample.text, chunk_size=80, overlap=15, strategy="sliding_window")
show_chunking_comparison(sample, whole, windows)

## 1.2 - Embeddings and the simple vector database

An **embedding** maps text to a vector so that *similar meanings land near each other*. Once every
chunk is a vector, "search" becomes "find the nearest vectors to the query vector".

You'll wire up the full ingestion step on `IngestionCore`: chunk the document, wrap each passage in
a `Chunk`, embed them all in one batched call, and store them. The `embedder`, `db`, and chunking
config are already on `self`.

- `chunking.chunk_text(text, chunk_size=..., overlap=..., strategy=...) -> list[str]`
- `Chunk(document_id=..., index=..., text=..., metadata=...)`
- `self.embedder.embed_batch(list_of_texts) -> list[list[float]]` (order preserved)
- `self.db.ensure_collection(name, vector_size=...)` then `self.db.insert_document(name, document, chunks)`

### 🎯 Exercise 2 - implement `ingest_document`

In [ ]:
def ingest_document(self, document):
    """Run load->CHUNK->EMBED->store for one Document. Return the list of stored Chunks."""
    # 🎯 TODO: split the document's text into chunk texts, using the chunking
    # settings already stored on self (self.chunk_size, self.overlap, self.strategy)
    texts = chunking.chunk_text(...)

    # 🎯 TODO: if chunking produced nothing (e.g. an empty document), there is
    # nothing to embed or store, return an empty list right away
    ...

    # 🎯 TODO: wrap each chunk text in a Chunk object. Every chunk needs:
    #   - document_id: which document it came from (document.id)
    #   - index: its position among this document's chunks (use enumerate)
    #   - text: the chunk text itself
    #   - metadata: a copy of the document's own metadata, so every chunk
    #     remembers where it came from
    chunks = [
        ...
        for i, text in enumerate(texts)
    ]

    # 🎯 TODO: embed every chunk's text in a single batched call (much faster
    # than one call per chunk), then attach each vector back onto its chunk
    embeddings = self.embedder.embed_batch(...)
    for chunk, embedding in zip(chunks, embeddings):
        chunk.embedding = ...

    # 🎯 TODO: make sure the collection exists, sized to match the real
    # embedding length you just got back, then persist the document and
    # its chunks into the store
    self.db.ensure_collection(...)
    self.db.insert_document(...)

    return chunks


In [ ]:
IngestionCore.ingest_document = ingest_document
print("Patched IngestionCore.ingest_document")

In [ ]:
suite = TestSuite("Part 1.2: ingest_document")

def _fresh_core():
    db = DBManager()  # in-memory, ephemeral
    core = IngestionCore(MockEmbedder([1.0, 0.0, 0.0]), db, collection_name="t",
                         chunk_size=5, overlap=1)
    return db, core

_doc = Document(text="one two three four five six seven eight nine ten",
                metadata={"source": "x.md", "document_title": "X"})

@suite.case("ingest_document", "every chunk comes back embedded")
def _():
    _db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert chunks and all(c.embedding is not None for c in chunks)

@suite.case("ingest_document", "chunk indices are sequential from 0")
def _():
    _db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert [c.index for c in chunks] == list(range(len(chunks)))

@suite.case("ingest_document", "chunks are persisted and re-readable from the store")
def _():
    db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert len(db.get_all_chunks("t")) == len(chunks)

@suite.case("ingest_document", "document metadata rides onto every chunk")
def _():
    _db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert all(c.metadata.get("source") == "x.md" for c in chunks)

suite.run()

**Where did that chunk actually go?** Every chunk you just embedded becomes one row in the
Qdrant collection: its vector, its text, and its metadata, all stored together.

In [ ]:
#@title 📊 Diagram: chunk to vector to Qdrant row (run me to view) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div style="font-family:ui-sans-serif,system-ui,sans-serif;max-width:760px;margin:10px auto">
<svg viewBox="0 0 760 330" width="100%" style="max-width:760px">
<defs>
  <marker id="ahb" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
    <path d="M0,0 L8,3 L0,6 Z" fill="#647084"/>
  </marker>
</defs>

<rect x="10" y="10" width="230" height="72" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/>
<text x="125" y="34" text-anchor="middle" font-size="11.5" font-weight="700" fill="#1f2933">chunk.text</text>
<text x="125" y="52" text-anchor="middle" font-size="10" fill="#647084">"Employees receive 25 days</text>
<text x="125" y="66" text-anchor="middle" font-size="10" fill="#647084">of annual leave per year."</text>

<path d="M240,46 L280,46" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahb)"/>

<rect x="285" y="10" width="230" height="72" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/>
<text x="400" y="34" text-anchor="middle" font-size="11.5" font-weight="700" fill="#1f2933">embedding vector</text>
<text x="400" y="53" text-anchor="middle" font-size="10" font-family="ui-monospace,monospace" fill="#647084">
  [0.12, -0.07, 0.88, ...]
</text>
<text x="400" y="68" text-anchor="middle" font-size="9.5" fill="#b45309" font-weight="600">
  3072 numbers (real model)
</text>

<path d="M515,46 L555,46" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahb)"/>

<rect x="560" y="10" width="190" height="72" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/>
<text x="655" y="34" text-anchor="middle" font-size="11.5" font-weight="700" fill="#1f2933">one Qdrant row</text>
<text x="655" y="52" text-anchor="middle" font-size="9.5" fill="#647084">id, vector, text,</text>
<text x="655" y="66" text-anchor="middle" font-size="9.5" fill="#647084">metadata</text>

<text x="10" y="112" font-size="10.5" fill="#647084">
  In this notebook's offline exercises, MockEmbedder returns a tiny 3-number vector instead,
  same idea, just small enough to eyeball.
</text>

<text x="10" y="150" font-size="12" font-weight="700" fill="#334155">the collection fills up, one row per chunk</text>
<rect x="10" y="164" width="740" height="26" fill="#f8fafc" stroke="#e3e8ef"/>
<text x="20" y="181" font-size="10" font-weight="700" fill="#334155">id</text>
<text x="120" y="181" font-size="10" font-weight="700" fill="#334155">vector</text>
<text x="300" y="181" font-size="10" font-weight="700" fill="#334155">text</text>
<text x="560" y="181" font-size="10" font-weight="700" fill="#334155">metadata</text>

<rect x="10" y="190" width="740" height="26" fill="#ffffff" stroke="#e3e8ef"/>
<text x="20" y="207" font-size="9.5" font-family="ui-monospace,monospace" fill="#647084">a1b2...</text>
<text x="120" y="207" font-size="9.5" font-family="ui-monospace,monospace" fill="#647084">[0.12, -0.07, ...]</text>
<text x="300" y="207" font-size="9.5" fill="#1f2933">"Employees receive 25 days..."</text>
<text x="560" y="207" font-size="9.5" fill="#647084">source: leave_policy.md</text>

<rect x="10" y="216" width="740" height="26" fill="#f8fafc" stroke="#e3e8ef"/>
<text x="20" y="233" font-size="9.5" font-family="ui-monospace,monospace" fill="#647084">c3d4...</text>
<text x="120" y="233" font-size="9.5" font-family="ui-monospace,monospace" fill="#647084">[0.31, 0.02, ...]</text>
<text x="300" y="233" font-size="9.5" fill="#1f2933">"Remote work requires..."</text>
<text x="560" y="233" font-size="9.5" fill="#647084">source: remote_work.md</text>

<rect x="10" y="242" width="740" height="26" fill="#ffffff" stroke="#e3e8ef"/>
<text x="20" y="259" font-size="9.5" font-family="ui-monospace,monospace" fill="#647084">e5f6...</text>
<text x="120" y="259" font-size="9.5" font-family="ui-monospace,monospace" fill="#647084">[-0.09, 0.44, ...]</text>
<text x="300" y="259" font-size="9.5" fill="#1f2933">"Vendors undergo a risk..."</text>
<text x="560" y="259" font-size="9.5" fill="#647084">source: procurement.md</text>

<text x="20" y="285" font-size="14" fill="#647084">...</text>
<text x="10" y="310" font-size="10.5" fill="#647084">
  every document you ingest adds more rows, this is what hybrid_search() later scans through.
</text>
</svg>
</div>
'''))


## 1.3 - Metadata

When we load a file we attach **metadata**: where it came from (`source`) and a human title
(`document_title`). That metadata travels with every chunk, and later lets retrieval *filter*,
for example "only search the leave policy", so unrelated documents can't crowd out the right answer.

You'll implement the title-deriving helper: use the first Markdown `# heading` if there is one,
otherwise fall back to a tidied-up file stem (`leave_policy` becomes `"Leave Policy"`).

### 🎯 Exercise 3 - implement `derive_title`

In [ ]:
def derive_title(text, stem):
    """Return a document title: first Markdown heading, else a tidied file stem."""
    # 🎯 TODO: go through the text one line at a time, looking for the first
    # Markdown heading (a line that starts with "# " after trimming whitespace)
    for line in text.splitlines():
        stripped = ...
        if ...:
            # 🎯 TODO: return the heading text itself, with the leading "# "
            # removed and any extra whitespace trimmed off
            return ...

    # 🎯 TODO: no heading was found, fall back to the file stem, tidied up:
    # replace underscores and hyphens with spaces, then title-case it
    # (e.g. "leave_and_absence" -> "Leave And Absence")
    return ...


In [ ]:
loaders._derive_title = derive_title
print("Patched loaders._derive_title")

In [ ]:
suite = TestSuite("Part 1.3: derive_title")

@suite.case("derive_title", "uses the first Markdown heading when present")
def _():
    assert derive_title("# Leave Policy\n\nText...", "leave_policy") == "Leave Policy"

@suite.case("derive_title", "falls back to a title-cased stem")
def _():
    assert derive_title("no heading here", "leave_and_absence") == "Leave And Absence"

suite.run()

**Metadata makes search precise.** Here is the payoff: given a pile of chunks from *all* policies,
a metadata filter narrows the field to one document *before* any scoring happens. (The filter
helper is provided, you'll use it for real in Part 2.)

In [ ]:
all_chunks = []
_db = DBManager()
_core = IngestionCore(MockEmbedder([1.0, 0.0, 0.0]), _db, collection_name="demo",
                      chunk_size=120, overlap=20)
for d in docs:
    all_chunks.extend(_core.ingest_document(d))

only_leave = RetrievalCore._apply_metadata_filter(all_chunks, {"source": "leave_and_absence_policy.md"})
display(HTML(_card(
    f'<b style="color:{_INK}">Metadata filter</b> '
    f'<code style="color:{_TEAL}">{{"source": "leave_and_absence_policy.md"}}</code><br>'
    f'corpus: <b>{len(all_chunks)}</b> chunks &nbsp;→&nbsp; '
    f'after filter: <b style="color:{_TEAL}">{len(only_leave)}</b> chunks')))
show_chunks(only_leave, title="Leave-policy chunks (after filtering)")

## 1.4 - Export your Part 1 functions

This writes your three functions to `solutions/part1_ingestion.py`. Dropped into the repo, the app
will run on *your* ingestion code (see Part 3). **Run your implementation cells above first**, then
run this.

In [ ]:
HEADER_P1 = (
    '"""Part 1, exported from the notebook. Do not edit by hand, re-export instead."""\n'
    'from __future__ import annotations\n'
    'import chunking\n'
    'from models import Chunk\n\n\n'
)

_funcs_p1 = [sliding_window, ingest_document, derive_title]
_body = HEADER_P1 + "\n\n".join(inspect.getsource(f) for f in _funcs_p1)

_path = pathlib.Path("solutions/part1_ingestion.py")
_path.parent.mkdir(exist_ok=True)
_path.write_text(_body)
print(f"Wrote {_path} ({len(_funcs_p1)} functions).")

try:
    from google.colab import files
    files.download(str(_path))
except Exception:
    pass

### Bonus (optional): smarter vector stores

The provided store keeps chunks in a flat list and compares the query against *every* vector.
That's fine for a few thousand chunks but doesn't scale. Real systems use **approximate
nearest-neighbour** indexes such as **HNSW** (Hierarchical Navigable Small World graphs), which
trade a tiny bit of accuracy for enormous speed. Qdrant (the store under `DBManager`) uses HNSW
internally. *Bonus:* read how HNSW builds a layered graph and why it's so much faster than a linear
scan.

---
# Part 2 - The RAG core

Now the **read side**: take a question and produce a grounded answer.

You'll build each retrieval strategy (the ✏️ steps), combine them, optionally rerank, and finally
hand the chosen chunks to the LLM.

In [ ]:
#@title 📊 Diagram: Part 2 flow (run me to view) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 900 214" width="100%" style="max-width:900px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ahp2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><rect x="16" y="82" width="124" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="78.0" y="114.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Question</text><rect x="190" y="18" width="168" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="274.0" y="41" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Keyword search</text><text x="274.0" y="58" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">BM25  &#9999;&#65039;</text><rect x="190" y="138" width="168" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="274.0" y="161" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Embedding search</text><text x="274.0" y="178" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">cosine  &#9999;&#65039;</text><rect x="400" y="82" width="116" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="458.0" y="105" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Fuse</text><text x="458.0" y="122" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">RRF  &#9999;&#65039;</text><rect x="548" y="82" width="116" height="54" rx="10" fill="#ffffff" stroke="#e3e8ef" stroke-width="1.6"/><text x="606.0" y="105" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Rerank</text><text x="606.0" y="122" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">bonus</text><rect x="696" y="82" width="92" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="742.0" y="114.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">LLM</text><rect x="800" y="82" width="92" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="846.0" y="114.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Answer</text><path d="M140,100 L190,58" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/><path d="M140,114 L190,178" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/><path d="M358,58 L400,100" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/><path d="M358,178 L400,120" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/><path d="M516,109 L548,109" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/><path d="M664,109 L696,109" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/><path d="M788,109 L800,109" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ahp2)"/></svg></div>
'''))


## 2.1 - Embedding (semantic) search

Semantic search ranks chunks by how close their vectors are to the **query's** vector. The standard
closeness measure is **cosine similarity**, the angle between two vectors, ignoring their length:

```
cos(q, d) = (q . d) / (||q|| * ||d||)
```

You'll implement it in a vectorised way: `query_vec` is one vector, `matrix` has one chunk vector
per row, and you return one score per row. Guard against divide-by-zero (a zero vector gives score 0).

### 🎯 Exercise 4 - implement `cosine_similarity`

In [ ]:
#@title 📊 Diagram: cosine similarity angle (run me to view) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 520 210" width="100%" style="max-width:520px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ahc" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><line x1="70" y1="172" x2="234.6" y2="67.8" stroke="#0f766e" stroke-width="2.4" marker-end="url(#ahc)"/><line x1="70" y1="172" x2="274.8" y2="125.1" stroke="#b45309" stroke-width="2.4" marker-end="url(#ahc)"/><path d="M110.9,162.6 A 42,42 0 0 1 105.5,149.6" fill="none" stroke="#647084" stroke-width="1.6"/><text x="128" y="150" text-anchor="middle" font-size="14" font-weight="700" fill="#1f2933">&#952;</text><text x="240" y="60" text-anchor="start" font-size="12" font-weight="700" fill="#0f766e">query vector q</text><text x="280" y="127" text-anchor="start" font-size="12" font-weight="700" fill="#b45309">chunk vector d</text><text x="60" y="192" text-anchor="start" font-size="11" font-weight="600" fill="#334155">cos(q, d) = (q &#183; d) / (&#8214;q&#8214; &#8214;d&#8214;), small angle means score near 1</text></svg></div>
'''))


In [ ]:
def cosine_similarity(query_vec, matrix):
    """Cosine similarity of `query_vec` against every row of `matrix`."""
    # 🎯 TODO: work out the length ("norm") of every chunk vector (one number
    # per row of `matrix`), and multiply it by the length of the query vector.
    # This gives you one denominator per chunk.
    # (hint: np.linalg.norm(matrix, axis=1) gives the length of each row)
    denom = ...

    # 🎯 TODO: work out the dot product of the query vector against every row
    # of `matrix` in one go. The @ operator does matrix multiplication, which
    # is exactly "dot product against every row at once" here.
    scores = ...

    # 🎯 TODO: divide scores by denom to get the cosine similarity, but avoid
    # dividing by zero (a chunk vector that's all zeros would otherwise crash
    # this). np.divide's `out=` and `where=` arguments let you say "put 0 here
    # instead of dividing" wherever the denominator is 0.
    return np.divide(scores, denom, out=np.zeros_like(scores), where=...)


In [ ]:
# _cosine_similarity is a @staticmethod, so wrap it when patching.
RetrievalCore._cosine_similarity = staticmethod(cosine_similarity)
print("Patched RetrievalCore._cosine_similarity")

In [ ]:
suite = TestSuite("Part 2.1: cosine_similarity")

@suite.case("cosine_similarity", "identical direction -> 1.0")
def _():
    q = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    M = np.array([[1, 0, 0], [2, 0, 0]], dtype=np.float32)
    out = cosine_similarity(q, M)
    assert np.allclose(out, [1.0, 1.0])

@suite.case("cosine_similarity", "orthogonal -> 0.0")
def _():
    q = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    M = np.array([[0, 1, 0]], dtype=np.float32)
    assert np.isclose(cosine_similarity(q, M)[0], 0.0)

@suite.case("cosine_similarity", "zero vector is safe (no NaN)")
def _():
    q = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    M = np.array([[0, 0, 0]], dtype=np.float32)
    assert cosine_similarity(q, M)[0] == 0.0

suite.run()

## 2.2 - Keyword (lexical) search

Embeddings capture meaning but can miss an exact term: a specific article number, an acronym, a
rare word. **Keyword search** complements them by scoring exact token overlap. We use **BM25**, a
classic ranking function that rewards query terms appearing in a chunk while down-weighting words
that are common across the whole corpus.

The helpers are provided: `_tokenize(text)` (lowercase + split), `self._apply_metadata_filter(...)`,
and `self._top_k(candidates, scores, top_k)`. You wire them together with `BM25Okapi`.

### 🎯 Exercise 5 - implement `keyword_search`

In [ ]:
def keyword_search(self, query, top_k=None, metadata_filter=None):
    """Rank self.chunks by BM25 overlap with `query`. Return list[(Chunk, score)]."""
    # 🎯 TODO: fall back to self.top_k when the caller didn't pass one
    top_k = ...

    # 🎯 TODO: tokenize the query into a list of lowercase words
    query_tokens = _tokenize(...)

    # 🎯 TODO: narrow self.chunks down using the metadata filter (a provided
    # helper handles the actual matching, you just call it)
    candidates = self._apply_metadata_filter(...)

    # 🎯 TODO: with nothing to search, or nothing to search for, there is
    # nothing to return, bail out early with an empty list
    if ...:
        return []

    # 🎯 TODO: tokenize every candidate chunk's text the same way as the query,
    # this list of token lists is what BM25Okapi needs to build its index
    corpus = [... for chunk in candidates]

    # 🎯 TODO: build the BM25 index over this corpus, then score the query
    # against every chunk in it
    bm25 = BM25Okapi(...)
    scores = bm25.get_scores(...)

    # 🎯 TODO: pick the top_k highest-scoring chunks (a provided helper does
    # the sorting and pairing for you)
    return self._top_k(...)


In [ ]:
RetrievalCore.keyword_search = keyword_search
print("Patched RetrievalCore.keyword_search")

In [ ]:
suite = TestSuite("Part 2.2: keyword_search")

_kw_chunks = [
    Chunk(document_id="d", index=0, text="employees get twenty five days of annual leave",
          metadata={"source": "leave.md"}),
    Chunk(document_id="d", index=1, text="expenses must be submitted within thirty days",
          metadata={"source": "expense.md"}),
]
_rc = RetrievalCore(MockEmbedder(), chunks=_kw_chunks)

@suite.case("keyword_search", "the chunk with the query terms ranks first")
def _():
    res = _rc.keyword_search("how many days of annual leave", top_k=2)
    assert res[0][0].index == 0

@suite.case("keyword_search", "top_k caps the number of results")
def _():
    assert len(_rc.keyword_search("days", top_k=1)) == 1

@suite.case("keyword_search", "a metadata filter restricts the candidates")
def _():
    res = _rc.keyword_search("days", metadata_filter={"source": "expense.md"})
    assert all(c.metadata["source"] == "expense.md" for c, _ in res)

@suite.case("keyword_search", "empty query -> no results")
def _():
    assert _rc.keyword_search("   ") == []

suite.run()

## 2.3 - Combining the two: Reciprocal Rank Fusion

Keyword and embedding search return scores on **incompatible scales** (BM25 vs. cosine), so you
can't just add them. **Reciprocal Rank Fusion (RRF)** sidesteps this by using only each result's
*rank*:

```
fused_score(chunk) = sum over lists of  1 / (k + rank_in_that_list)     # rank is 0-based, k = 60
```

A chunk that ranks high in *either* list scores well; a chunk near the top of *both* wins. `k=60`
is the conventional dampening constant. Dedupe by `chunk.id` (a chunk can appear in both lists).

### 🎯 Exercise 6 - implement `reciprocal_rank_fusion`

In [ ]:
#@title 📊 Diagram: Reciprocal Rank Fusion example (run me to view) { display-mode: "form" }
from IPython.display import display, HTML

display(HTML(r'''
<div style="display:flex;gap:14px;align-items:center;justify-content:center;flex-wrap:wrap;margin:12px 0;font-family:ui-sans-serif,system-ui,sans-serif"><div style="border:1px solid #e3e8ef;border-top:3px solid #0f766e;border-radius:8px;padding:8px 12px;min-width:130px"><div style="font-size:11px;color:#0f766e;font-weight:700;text-transform:uppercase;letter-spacing:.04em;margin-bottom:2px">keyword</div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#1</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk A</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#2</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk B</span></div></div><div style="border:1px solid #e3e8ef;border-top:3px solid #0f766e;border-radius:8px;padding:8px 12px;min-width:130px"><div style="font-size:11px;color:#0f766e;font-weight:700;text-transform:uppercase;letter-spacing:.04em;margin-bottom:2px">embedding</div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#1</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk B</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#2</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk C</span></div></div><div style="font-size:24px;color:#647084">&#8594;</div><div style="border:1px solid #e3e8ef;border-top:3px solid #b45309;border-radius:8px;padding:8px 12px;min-width:130px"><div style="font-size:11px;color:#b45309;font-weight:700;text-transform:uppercase;letter-spacing:.04em;margin-bottom:2px">fused (RRF)</div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#1</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk B &#183; 1/60 + 1/61</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#2</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk A &#183; 1/60</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#3</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk C &#183; 1/61</span></div></div></div>
<div style="text-align:center;font-size:12px;color:#334155;margin-top:4px">
Above: <b>chunk B</b> is not first in either list, but it ranks well in <i>both</i>, so fusion floats it to the top,
exactly the behaviour we want.
</div>
'''))


In [ ]:
def reciprocal_rank_fusion(ranked_lists, top_k, k=60):
    """Fuse several ranked [(Chunk, score)] lists into one. Return list[(Chunk, fused_score)]."""
    # 🎯 TODO: two lookup tables you'll fill in below:
    #   fused_scores: chunk id -> running total score
    #   chunks_by_id: chunk id -> the actual Chunk object (so you can return it later)
    fused_scores = {}
    chunks_by_id = {}

    # 🎯 TODO: go through each ranked list you were given, and within each
    # list, go through its (chunk, score) pairs together with their rank
    # (enumerate gives you a 0-based position: rank 0 is the top result)
    for ranked in ranked_lists:
        for rank, (chunk, _score) in enumerate(ranked):
            # 🎯 TODO: add this chunk's contribution from this list,
            # 1 / (k + rank), onto its running total in fused_scores
            # (use .get(chunk.id, 0.0) so a chunk seen for the first time
            # starts from 0)
            fused_scores[chunk.id] = ...
            # 🎯 TODO: remember the actual chunk object for this id, so you
            # can look it up again once you have the final ranking
            chunks_by_id[chunk.id] = ...

    # 🎯 TODO: sort the chunk ids by their fused score, highest first
    ordered = sorted(fused_scores.items(), key=lambda item: item[1], reverse=True)

    # 🎯 TODO: take the top_k ids and turn each one back into a
    # (Chunk, fused_score) pair using chunks_by_id
    return [... for cid, score in ordered[:top_k]]


In [ ]:
RetrievalCore._reciprocal_rank_fusion = staticmethod(reciprocal_rank_fusion)
print("Patched RetrievalCore._reciprocal_rank_fusion")

In [ ]:
suite = TestSuite("Part 2.3: reciprocal_rank_fusion")

_a = Chunk(document_id="d", index=0, text="A")
_b = Chunk(document_id="d", index=1, text="B")
_c = Chunk(document_id="d", index=2, text="C")

@suite.case("reciprocal_rank_fusion", "a chunk ranked highly in both lists wins")
def _():
    list1 = [(_a, 9.0), (_b, 1.0)]   # b at rank 1
    list2 = [(_b, 9.0), (_c, 1.0)]   # b at rank 0  -> b appears in both
    fused = reciprocal_rank_fusion([list1, list2], top_k=3)
    assert fused[0][0].id == _b.id

@suite.case("reciprocal_rank_fusion", "results are deduplicated by chunk id")
def _():
    fused = reciprocal_rank_fusion([[(_a, 1.0)], [(_a, 1.0)]], top_k=5)
    assert len(fused) == 1

@suite.case("reciprocal_rank_fusion", "top_k limits the output")
def _():
    fused = reciprocal_rank_fusion([[(_a, 1.0), (_b, 1.0), (_c, 1.0)]], top_k=2)
    assert len(fused) == 2

suite.run()

With all three patched in, **hybrid search** (provided, it calls your functions) now works
end-to-end. Let's run a real query over the policy corpus we ingested earlier.

In [ ]:
_rc_all = RetrievalCore(MockEmbedder([1.0, 0.0, 0.0]), chunks=all_chunks, top_k=3)
show_results(_rc_all.hybrid_search("how many days of paternity leave"),
             title='Hybrid search · "how many days of paternity leave"')

## 2.4 - Reranking (bonus)

Fusion already gives you a ranked list. Reranking takes a second pass over just those results and
asks a simpler question for each one: **how many of the exact words in the question also appear in
this chunk?** Count the matches, that count becomes the chunk's new score, then sort so chunks with
more matching words float to the top.

Production systems use a **cross-encoder** that reads the query and each chunk *together*, which is
more accurate but slower. This word-overlap count is a lightweight stand-in that captures the same
idea. This exercise is optional and not required by the app.

### 🎯 Exercise 7 - implement `rerank` (bonus)

In [ ]:
def rerank(query, results, top_k=None):
    """Reorder [(Chunk, score)] by exact term overlap with the query (bonus)."""
    # 🎯 TODO: tokenize the query into a set of words, a set makes counting
    # "how many words are shared" a simple intersection below
    q_terms = set(_tokenize(...))

    def overlap(pair):
        # 🎯 TODO: count how many words this chunk's text shares with the
        # query. pair is (chunk, score), so the chunk's text is pair[0].text.
        # Tokenize it the same way, turn it into a set too, and intersect
        # with q_terms, the size of that intersection is the overlap count.
        return len(q_terms & set(_tokenize(...)))

    # 🎯 TODO: sort results by that overlap count, highest first
    reranked = sorted(results, key=overlap, reverse=True)
    return reranked if top_k is None else reranked[:top_k]


In [ ]:
suite = TestSuite("Part 2.4: rerank")

_r1 = Chunk(document_id="d", index=0, text="annual leave days for employees")
_r2 = Chunk(document_id="d", index=1, text="a completely unrelated sentence")
_r3 = Chunk(document_id="d", index=2, text="employees get annual leave days off")

@suite.case("rerank", "a chunk sharing more query words ranks higher")
def _():
    results = [(_r2, 0.5), (_r1, 0.4), (_r3, 0.3)]  # deliberately out of the "right" order
    out = rerank("annual leave days", results)
    assert out[0][0].id == _r3.id or out[0][0].id == _r1.id
    assert out[-1][0].id == _r2.id

@suite.case("rerank", "top_k caps the number of results")
def _():
    results = [(_r1, 0.4), (_r2, 0.5), (_r3, 0.3)]
    assert len(rerank("annual leave days", results, top_k=2)) == 2

@suite.case("rerank", "top_k=None keeps every result")
def _():
    results = [(_r1, 0.4), (_r2, 0.5), (_r3, 0.3)]
    assert len(rerank("annual leave days", results)) == 3

suite.run()


## 2.5 - Putting it together: answer with the LLM

The final step of RAG: **retrieve**, stuff the chosen chunk texts into a **context block**, and ask
the LLM to answer *using only that context*. You return both the answer and the source chunks (the
web app shows them as citations).

Provided on `self`: `self._retrieve(query, search_type, metadata_filter)` (dispatches to the right
search), `self.llm.complete(prompt, system_prompt=...)`, and `self.system_prompt`.

### 🎯 Exercise 8 - implement `retrieve_and_answer`

In [ ]:
def retrieve_and_answer(self, query, search_type=None, metadata_filter=None):
    """Retrieve, then answer with the LLM. Return (answer_str, list[(Chunk, score)])."""
    # 🎯 TODO: retrieve the relevant chunks for this query (a provided helper
    # picks the right search strategy for you)
    results = self._retrieve(...)

    # 🎯 TODO: join every retrieved chunk's text into one block of context,
    # with a blank line between chunks so they stay visually separated
    context = "\n\n".join(...)

    # 🎯 TODO: build the final prompt: the context block, then the question,
    # following the exact template "Context:\n{context}\n\nQuestion: {query}"
    prompt = ...

    # 🎯 TODO: ask the LLM to answer, using the notebook's system prompt so it
    # sticks to only the given context
    answer = self.llm.complete(...)

    return answer, results


In [ ]:
RAGCore.retrieve_and_answer = retrieve_and_answer
print("Patched RAGCore.retrieve_and_answer")

In [ ]:
suite = TestSuite("Part 2.5: retrieve_and_answer")

# A full RAGCore wired to mocks: no network, fully deterministic.
_rag = RAGCore(MockEmbedder([1.0, 0.0, 0.0]), MockLLM(), chunk_size=5, overlap=1)
_rag.ingest_document(Document(
    text="employees are entitled to twenty five days of annual leave per year",
    metadata={"source": "leave.md", "document_title": "Leave"},
))

@suite.case("retrieve_and_answer", "returns an answer plus the source chunks")
def _():
    answer, sources = _rag.retrieve_and_answer("annual leave days", search_type=SearchType.KEYWORD)
    assert isinstance(answer, str) and answer
    assert len(sources) >= 1

@suite.case("retrieve_and_answer", "the retrieved context reaches the LLM prompt")
def _():
    _rag.retrieve_and_answer("annual leave days", search_type=SearchType.KEYWORD)
    assert "annual leave" in _rag.llm.last_prompt   # MockLLM records what it was sent

suite.run()

**See it rendered.** The mock LLM just echoes the prompt it received, so this view doubles as an
X-ray of *exactly* what your pipeline sends the model: the retrieved context plus the question.

In [ ]:
_ans, _src = _rag.retrieve_and_answer("how many days of annual leave", search_type=SearchType.KEYWORD)
show_answer(_ans, _src)

### Live demo (optional, needs an OpenRouter key)

If you set a key in 0.2, this ingests the real policy folder and answers a question with the actual
embedding and LLM models. Otherwise it is skipped.

In [ ]:
if HAS_KEY:
    from clients.embedder import TextEmbedder
    from clients.llm import LLMClient
    live = RAGCore(TextEmbedder(api_key=os.environ["OPENROUTER_API_KEY"]),
                   LLMClient(api_key=os.environ["OPENROUTER_API_KEY"]))
    live.ingest_path("data")
    answer, sources = live.retrieve_and_answer(
        "Can I accept a CHF 120 gift from a vendor?", search_type=SearchType.HYBRID)
    show_answer(answer, sources)
else:
    print("No API key set. Skipping the live demo (the offline tests above already prove it works).")

## 2.6 - Export your Part 2 functions

Writes your retrieval functions to `solutions/part2_retrieval.py`. **Run your implementation cells
above first.**

In [ ]:
HEADER_P2 = (
    '"""Part 2, exported from the notebook. Do not edit by hand, re-export instead."""\n'
    'from __future__ import annotations\n'
    'import numpy as np\n'
    'from rank_bm25 import BM25Okapi\n'
    'from retrieval_core import _tokenize\n\n\n'
)

_funcs_p2 = [cosine_similarity, keyword_search, reciprocal_rank_fusion, retrieve_and_answer]
_body = HEADER_P2 + "\n\n".join(inspect.getsource(f) for f in _funcs_p2)

_path = pathlib.Path("solutions/part2_retrieval.py")
_path.parent.mkdir(exist_ok=True)
_path.write_text(_body)
print(f"Wrote {_path} ({len(_funcs_p2)} functions).")

try:
    from google.colab import files
    files.download(str(_path))
except Exception:
    pass

---
# Part 3 - Run the full application (optional)

You've built the engine. The repo also ships a small **web app** around it so you can try your RAG
system in a browser, including uploading your own documents through the UI.

## How it fits together

- **Backend** (`server.py`, FastAPI): wraps your pipeline in an HTTP API. It holds one shared
  persistent vector store and builds a fresh `RAGCore` per request. Each request carries *your own*
  OpenRouter key in a header, the server never stores it.
- **Frontend** (`frontend/`, React + Vite): a key field, document upload, a question box with a
  strategy selector, and an answer view with expandable source citations.
- **Your code plugs in via `solutions.apply()`**: on startup the app monkey-patches the two files
  you exported (`solutions/part1_ingestion.py`, `solutions/part2_retrieval.py`) onto the real
  modules, exactly what you did cell-by-cell here. If a file is missing the app falls back to the
  reference implementation, so it always runs.

## Steps

1. Make sure your exported files are in the repo's `solutions/` folder (Parts 1.4 and 2.6).
2. **Backend:**
   ```bash
   uv sync                       # or: pip install -e .
   uv run uvicorn server:app --reload
   ```
3. **Frontend** (in another terminal):
   ```bash
   cd frontend && npm install && npm run dev
   ```
4. Open the printed URL, paste your OpenRouter key, upload the policies in `data/`, and ask away.

> The backend prints `[solutions] applied student implementations: ...` at startup when it picks up
> your exported files, a quick way to confirm your code is the one running.

## 3.1 - Live UI inside the notebook

Once both servers are running (see the instructions above), the cell below embeds the full
web app directly here. Re-run it at any time to reconnect.

In [ ]:
#@title 🖥️ 3.1 - Embed the live app (local mode only) { display-mode: "form" }

import urllib.request
from IPython.display import HTML, IFrame, display

BACKEND_URL  = "http://localhost:8000"
FRONTEND_URL = "http://localhost:5173"

_INK, _TEAL, _AMBER = "#1f2933", "#0f766e", "#b45309"
_LINE, _BG, _MUTE   = "#e3e8ef", "#f8fafc", "#647084"
_FONT = "ui-sans-serif,system-ui,sans-serif"
_MONO = "ui-monospace,SFMono-Regular,Menlo,monospace"

def _card(body, accent=_TEAL):
    return (f'<div style="border:1px solid {_LINE};border-left:4px solid {accent};'
            f'border-radius:10px;padding:16px 20px;margin-bottom:8px;background:#fff;'
            f'font-family:{_FONT}">{body}</div>')

def _code_block(label, code):
    return (f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;'
            f'padding:10px 14px;margin:6px 0">'
            f'<div style="font-size:11px;color:{_TEAL};font-weight:700;margin-bottom:4px">{label}</div>'
            f'<code style="font-size:13px;color:{_INK};font-family:{_MONO}">{code}</code></div>')

if IN_COLAB:
    display(HTML(_card(
        f'<b>Colab detected.</b> Skip this cell, run <b>3.2</b> below instead.',
        accent=_AMBER
    )))
else:
    try:
        with urllib.request.urlopen(f"{BACKEND_URL}/health", timeout=2) as r:
            backend_ok = r.status == 200
    except Exception:
        backend_ok = False

    if not backend_ok:
        display(HTML(_card(
            f'<div style="font-weight:700;color:{_INK};font-size:15px;margin-bottom:10px">'
            f'Backend not reachable at <code>{BACKEND_URL}</code></div>'
            f'<p style="color:{_MUTE};font-size:13px;margin:0 0 12px">'
            f'Start both processes from the repo root, then <b>re-run this cell</b>.</p>'
            + _code_block("Terminal 1: backend", "uv run uvicorn server:app --reload")
            + _code_block("Terminal 2: frontend", "cd frontend &amp;&amp; npm install &amp;&amp; npm run dev"),
            accent=_AMBER
        )))
    else:
        display(HTML(_card(
            f'<div style="display:flex;align-items:center;gap:10px">'
            f'<span style="font-size:18px">🟢</span>'
            f'<span style="font-weight:700;color:{_INK}">Backend connected</span>'
            f'<code style="color:{_MUTE};font-size:12px;font-family:{_MONO}">{BACKEND_URL}</code>'
            f'<span style="margin-left:auto;font-size:12px;color:{_MUTE}">'
            f'frontend → {FRONTEND_URL}</span></div>'
        )))
        display(IFrame(src=FRONTEND_URL, width="100%", height=780))

In [ ]:
#@title 🖥️ 3.2 - Build and serve the React frontend inside Colab { display-mode: "form" }

import os, subprocess, threading, time, requests
from IPython.display import IFrame, display

if not IN_COLAB:
    print("This cell is Colab-only. Use cell 3.1 above for local mode.")
else:
    os.chdir("/content/WE6_ProjectRAG")

    # ── 1. Get proxy URL FIRST (needed for the build) ─────────────────────────
    from google.colab.output import eval_js
    PROXY = str(eval_js("google.colab.kernel.proxyPort(8000)"))
    print(f"✓  Proxy URL: {PROXY}")

    # ── 2. Build frontend with VITE_API_BASE = proxy URL ──────────────────────
    FRONTEND_DIR = os.path.join(os.getcwd(), "frontend")

    print("Installing npm dependencies…")
    subprocess.run(["npm", "install"], cwd=FRONTEND_DIR, check=True, capture_output=True)

    print("Building frontend…")
    subprocess.run(
        ["npm", "run", "build"],
        cwd=FRONTEND_DIR,
        env={**os.environ, "VITE_API_BASE": PROXY},
        check=True,
    )
    print("✓  Frontend built")

    # ── 3. Start backend (serves API + frontend static files) ─────────────────
    if "_rag_backend_started" not in globals():
        import uvicorn

        def _run_backend():
            uvicorn.run("server:app", host="0.0.0.0", port=8000, log_level="info")

        threading.Thread(target=_run_backend, daemon=True).start()
        time.sleep(3)
        globals()["_rag_backend_started"] = True
        print("✓  Backend started on port 8000")
    else:
        print("✓  Backend already running on port 8000")

    # ── 4. Verify ─────────────────────────────────────────────────────────────
    try:
        r = requests.get("http://localhost:8000/health", timeout=5)
        print(f"  Health: {r.json()}")
    except Exception as e:
        print(f"  ⚠️  Backend check failed: {e}")

    # ── 5. Embed ──────────────────────────────────────────────────────────────
    display(IFrame(src=PROXY, width="100%", height=780))
    print("✓  App rendered")

---
# Bonus (optional) - find where this system breaks

Every RAG system has blind spots. Now that you understand chunking, embeddings, fusion, and
reranking from the inside, use that understanding to break your own system on purpose.

Try questions like:

- A question whose real answer is split across **two different policies** with numbers that
  almost, but don't quite, agree.
- A question about something that **isn't in any document at all**. Does the system say so, or
  does it make something up?
- A question phrased with **no shared vocabulary** with the source text (a synonym-heavy rewrite
  of a policy sentence). Does semantic search still find it?
- A question that is technically **answerable from a chunk boundary**, split so the key sentence
  gets cut in half. Does the overlap save you, or does it fail anyway?

Use the cell below to ask a few of these against the offline pipeline (built from your own
functions, no API key needed) and see what actually happens.

In [ ]:
# An offline RAGCore built from your own functions, over the real policy corpus.
_bonus_rag = RAGCore(MockEmbedder([1.0, 0.0, 0.0]), MockLLM(), chunk_size=120, overlap=20)
for _d in docs:
    _bonus_rag.ingest_document(_d)

# 🎯 TODO: try your own tricky questions here
_bonus_answer, _bonus_sources = _bonus_rag.retrieve_and_answer(
    "How many days of parental leave do contractors get in our Zurich office?",
    search_type=SearchType.HYBRID,
)
show_answer(_bonus_answer, _bonus_sources)


> **Your verdict - this system's failure modes**
>
> **Verdict:** Trustworthy / Trustworthy with caveats / Not ready  *(keep one)*
>
> **Why (one sentence):**

---
# Self-reflection

A handful of open questions, there's no single right answer, the point is to notice what you now
understand that you didn't before.

1. If you doubled `chunk_size` without changing `overlap`, what would you expect to happen to
   retrieval quality, and why?
2. Embeddings and keyword search each catch things the other misses. Give one concrete example of a
   question where you'd expect keyword search to win, and one where you'd expect embedding search
   to win.
3. Why does Reciprocal Rank Fusion use each result's *rank* instead of its raw score?
4. Reranking only reorders the chunks fusion already picked, it can never pull in a chunk that
   wasn't retrieved at all. What does that limit tell you about where reranking helps and where it
   can't?
5. If you were deploying this system for real employees asking about real policies, what's the one
   change you'd make first, and why?

---
# What you've learned

You built the core of a working RAG system, end to end:

- **Chunking**: why a whole document can't be a single embedding, and how a sliding window with
  overlap keeps sentences from being cut in half.
- **Embeddings and metadata**: how text becomes a vector, how that vector gets stored alongside
  its source document, and how metadata lets you filter before you even score anything.
- **Semantic search**: cosine similarity, and why it measures *direction* rather than length.
- **Keyword search**: BM25, and why exact terms still matter even when you have embeddings.
- **Fusion**: how Reciprocal Rank Fusion combines two incompatible scoring scales using only rank.
- **Reranking**: a cheap second pass that reorders results using exact term overlap.
- **Answering**: how retrieved chunks become a context block, and how a system prompt keeps the
  LLM grounded in that context instead of making things up.

Every one of these is a real design decision with real tradeoffs, not a fixed recipe. Now that
you've implemented it, you're in a position to argue about those tradeoffs, which is the actual
skill this project was teaching.

---
## Appendix - write your own tests

Use this scratch space to probe anything you're unsure about: print intermediate values, try edge
cases, or add `TestSuite` checks of your own. The same harness the graders use:

In [ ]:
suite = TestSuite("My experiments")

@suite.case("sliding_window", "my own edge case")
def _():
    # ✏️ change me
    assert sliding_window("a b c d", chunk_size=2, overlap=0) == ["a b", "c d"]

suite.run()

# ...or just scratch:
# print(chunking.chunk_text(docs[0].text, chunk_size=50, overlap=10)[:2])